# Locks and Deadlocks

## Overview
Locks are the mechanism databases use to enforce isolation between concurrent transactions. Understanding lock types, when they are acquired, and how deadlocks form is essential for building correct high-concurrency financial systems.

**Two levels of locking:**
- **Table-level locks** — protect the table structure; acquired automatically by DML and DDL
- **Row-level locks** — protect specific rows; acquired by `SELECT FOR UPDATE`, `INSERT`, `UPDATE`, `DELETE`

**The key row-level lock statements (PostgreSQL):**
```sql
SELECT ... FOR UPDATE          -- exclusive row lock; blocks all other writers
SELECT ... FOR SHARE           -- shared row lock; allows other FOR SHARE; blocks FOR UPDATE
SELECT ... FOR UPDATE NOWAIT   -- fail immediately if lock unavailable
SELECT ... FOR UPDATE SKIP LOCKED  -- skip locked rows; ideal for job queues
```

**What causes a deadlock:**
Two transactions each hold a lock the other needs. Neither can proceed. Neither will release. PostgreSQL detects the cycle (usually within 1 second) and aborts the transaction that has done less work, raising `ERROR: deadlock detected`.

**SQLite locking model:**
SQLite uses a single database-level write lock (not row-level). In WAL mode, readers never block writers and writers never block readers — but only one writer can proceed at a time.

---

In [1]:
import sqlite3

def get_conn():
    c = sqlite3.connect(":memory:")
    c.execute("PRAGMA journal_mode=WAL")
    c.row_factory = sqlite3.Row
    return c

conn = get_conn()
conn.executescript("""
CREATE TABLE accounts (
    account_id TEXT PRIMARY KEY,
    owner_name TEXT NOT NULL,
    balance    REAL NOT NULL CHECK(balance >= 0)
);
INSERT INTO accounts VALUES
    ('ACC001','Aroha Ngata', 5000.0),
    ('ACC002','Liam Chen',   2500.0),
    ('ACC003','Fatima Rashid',8000.0);
""")
conn.commit()

print("=== Lock types overview ===")
lock_types = [
    ("ROW SHARE (RowShareLock)",        "SELECT FOR SHARE",          "Blocks: EXCLUSIVE, ACCESS EXCLUSIVE"),
    ("ROW EXCLUSIVE (RowExclusiveLock)","INSERT/UPDATE/DELETE",      "Blocks: SHARE, SHARE ROW EXCLUSIVE, EXCLUSIVE, ACCESS EXCLUSIVE"),
    ("SHARE (ShareLock)",               "CREATE INDEX (non-concurrent)","Blocks: ROW EXCLUSIVE, SHARE ROW EXCLUSIVE, EXCLUSIVE, ACCESS EXCLUSIVE"),
    ("EXCLUSIVE (ExclusiveLock)",       "REFRESH MATERIALIZED VIEW", "Blocks everything except ACCESS SHARE"),
    ("ACCESS EXCLUSIVE",                "ALTER TABLE, DROP TABLE",   "Blocks everything — full table lock"),
    ("FOR UPDATE (row-level)",          "SELECT ... FOR UPDATE",     "Locks specific rows; blocks other FOR UPDATE"),
    ("FOR SHARE  (row-level)",          "SELECT ... FOR SHARE",      "Locks specific rows; allows other FOR SHARE"),
    ("FOR NO KEY UPDATE",               "UPDATE (no PK change)",     "Less restrictive than FOR UPDATE"),
    ("FOR KEY SHARE",                   "FK reference check",        "Weakest row lock; only blocks FOR UPDATE"),
]
print(f"  {'Lock type':<36s}  {'Acquired by':<36s}  Blocks")
print("  " + "-"*100)
for lock, acquired, blocks in lock_types:
    print(f"  {lock:<36s}  {acquired:<36s}  {blocks}")


=== Lock types overview ===
  Lock type                             Acquired by                           Blocks
  ----------------------------------------------------------------------------------------------------
  ROW SHARE (RowShareLock)              SELECT FOR SHARE                      Blocks: EXCLUSIVE, ACCESS EXCLUSIVE
  ROW EXCLUSIVE (RowExclusiveLock)      INSERT/UPDATE/DELETE                  Blocks: SHARE, SHARE ROW EXCLUSIVE, EXCLUSIVE, ACCESS EXCLUSIVE
  SHARE (ShareLock)                     CREATE INDEX (non-concurrent)         Blocks: ROW EXCLUSIVE, SHARE ROW EXCLUSIVE, EXCLUSIVE, ACCESS EXCLUSIVE
  EXCLUSIVE (ExclusiveLock)             REFRESH MATERIALIZED VIEW             Blocks everything except ACCESS SHARE
  ACCESS EXCLUSIVE                      ALTER TABLE, DROP TABLE               Blocks everything — full table lock
  FOR UPDATE (row-level)                SELECT ... FOR UPDATE                 Locks specific rows; blocks other FOR UPDATE
  FOR SHARE  (row-level) 

---
## SELECT FOR UPDATE: pessimistic row locking

In [2]:

print("=== SELECT FOR UPDATE: pessimistic locking in action ===")
for_update_code = """
-- Scenario: two tellers simultaneously process a withdrawal from ACC001
-- Without locking, both could read balance=$5000 and both approve $4000 withdrawal

-- Teller A (correct approach with FOR UPDATE):
BEGIN;
SELECT balance FROM accounts WHERE account_id = 'ACC001' FOR UPDATE;
-- ↑ Locks ACC001's row. Teller B must wait here until A commits/rolls back.

-- Application checks: balance ($5000) >= withdrawal ($4000)? Yes.
UPDATE accounts SET balance = balance - 4000 WHERE account_id = 'ACC001';
INSERT INTO ledger(account_id, amount, entry_type) VALUES ('ACC001', 4000, 'debit');
COMMIT;
-- Teller A releases lock. Teller B now reads balance=$1000.

-- Teller B (now unblocked):
-- Checks: balance ($1000) >= withdrawal ($4000)? No. Declines.
-- ↑ Correct — the CHECK constraint OR application logic handles this.

-- SKIP LOCKED: process a queue without blocking on locked rows
BEGIN;
SELECT * FROM trades
WHERE  status = 'pending'
ORDER BY created_at
LIMIT  1
FOR UPDATE SKIP LOCKED;   -- skip any row locked by another worker
-- Update the row we got, then commit
-- Multiple workers can process the queue concurrently with no contention
COMMIT;

-- NOWAIT: fail immediately instead of waiting for a lock
BEGIN;
SELECT balance FROM accounts WHERE account_id = 'ACC001'
FOR UPDATE NOWAIT;
-- Raises: ERROR: could not obtain lock on row in relation "accounts"
-- Use when the business logic should fail fast rather than queue
COMMIT;
"""
print(for_update_code)
print("SQLite equivalent: SQLite uses database-level write lock, not row-level.")
print("In SQLite WAL mode: first writer wins; second writer gets SQLITE_BUSY.")
print("Use Python threading.Lock() or queue patterns for SQLite concurrency control.")


=== SELECT FOR UPDATE: pessimistic locking in action ===

-- Scenario: two tellers simultaneously process a withdrawal from ACC001
-- Without locking, both could read balance=$5000 and both approve $4000 withdrawal

-- Teller A (correct approach with FOR UPDATE):
BEGIN;
SELECT balance FROM accounts WHERE account_id = 'ACC001' FOR UPDATE;
-- ↑ Locks ACC001's row. Teller B must wait here until A commits/rolls back.

-- Application checks: balance ($5000) >= withdrawal ($4000)? Yes.
UPDATE accounts SET balance = balance - 4000 WHERE account_id = 'ACC001';
INSERT INTO ledger(account_id, amount, entry_type) VALUES ('ACC001', 4000, 'debit');
COMMIT;
-- Teller A releases lock. Teller B now reads balance=$1000.

-- Teller B (now unblocked):
-- Checks: balance ($1000) >= withdrawal ($4000)? No. Declines.
-- ↑ Correct — the CHECK constraint OR application logic handles this.

-- SKIP LOCKED: process a queue without blocking on locked rows
BEGIN;
SELECT * FROM trades
WHERE  status = 'pending'
ORD

---
## Deadlocks: causes and prevention

In [4]:
print("=== Deadlocks: what they are and how they happen ===")
deadlock_scenario = """
  Deadlock scenario: two transfers, opposite directions

  Time  Transaction A (ACC001 → ACC002)      Transaction B (ACC002 → ACC001)
  ────────────────────────────────────────────────────────────────────────────
  T1    BEGIN
  T2    SELECT ... FROM accounts
        WHERE account_id='ACC001'
        FOR UPDATE                           ← A locks ACC001
  T3                                         BEGIN
  T4                                         SELECT ... FROM accounts
                                             WHERE account_id='ACC002'
                                             FOR UPDATE   ← B locks ACC002
  T5    SELECT ... FROM accounts
        WHERE account_id='ACC002'
        FOR UPDATE                           ← A waits for B to release ACC002
  T6                                         SELECT ... FROM accounts
                                             WHERE account_id='ACC001'
                                             FOR UPDATE   ← B waits for A to release ACC001
  T7    *** DEADLOCK *** PostgreSQL detects the cycle and aborts one transaction
        ERROR: deadlock detected
        DETAIL: Process A waits for ShareLock on transaction B;
                Process B waits for ShareLock on transaction A.
        HINT:   See server log for query details.
"""
print(deadlock_scenario)

print("=== Deadlock prevention strategies ===")
strategies = [
    ("Consistent lock ordering",
     "Always acquire locks in the same order across all transactions.",
     "Always lock lower account_id first: lock MIN(from,to) before MAX(from,to).",
     """-- Python:
  ids = sorted([from_id, to_id])
  cursor.execute("SELECT ... FOR UPDATE ORDER BY account_id",
                 (ids[0], ids[1]))
  # Both transactions now always lock ACC001 before ACC002 — deadlock impossible"""),
    ("Short transactions",
     "Keep transactions brief; release locks quickly.",
     "Do all computation outside the transaction; only hold locks during writes.",
     """-- Bad:  open transaction, call external API, then write
  -- Good: call API, then open transaction and write"""),
    ("NOWAIT / SKIP LOCKED",
     "Fail or skip instead of waiting; let application retry.",
     "Queue processors use SKIP LOCKED to grab only unlocked work items.",
     """SELECT ... FOR UPDATE NOWAIT     -- raises error immediately on lock conflict
  SELECT ... FOR UPDATE SKIP LOCKED  -- skips locked rows, grabs next available"""),
    ("Deadlock detection (automatic)",
     "PostgreSQL detects deadlocks automatically and aborts one transaction.",
     "Application must catch DeadlockDetected error and retry.",
     """except psycopg2.errors.DeadlockDetected:
      conn.rollback()
      # retry the transaction"""),
]
for name, principle, example, code in strategies:
    print(f"  Strategy:  {name}")
    print(f"  Principle: {principle}")
    print(f"  Example:   {example}")
    print(f"  Code:\n{code}")
    print()

=== Deadlocks: what they are and how they happen ===

  Deadlock scenario: two transfers, opposite directions

  Time  Transaction A (ACC001 → ACC002)      Transaction B (ACC002 → ACC001)
  ────────────────────────────────────────────────────────────────────────────
  T1    BEGIN
  T2    SELECT ... FROM accounts
        WHERE account_id='ACC001'
        FOR UPDATE                           ← A locks ACC001
  T3                                         BEGIN
  T4                                         SELECT ... FROM accounts
                                             WHERE account_id='ACC002'
                                             FOR UPDATE   ← B locks ACC002
  T5    SELECT ... FROM accounts
        WHERE account_id='ACC002'
        FOR UPDATE                           ← A waits for B to release ACC002
  T6                                         SELECT ... FROM accounts
                                             WHERE account_id='ACC001'
                                    

---
## Common Pitfalls

**1. Not acquiring locks in a consistent order**
The most common cause of deadlocks in financial systems is two transactions acquiring the same rows in different orders. Transaction A locks ACC001 then ACC002; Transaction B locks ACC002 then ACC001. The fix is always the same: sort the IDs and acquire locks in ascending order in every transaction. This one rule eliminates the vast majority of deadlocks.

**2. Holding locks across network calls or user interactions**
Opening a transaction, locking rows, making an HTTP call to a payment processor, then writing back can hold locks for seconds or minutes. Every other transaction touching those rows is blocked for that duration. Complete all external calls before opening the transaction; do only database writes inside it.

**3. Not catching DeadlockDetected in application code**
PostgreSQL automatically detects deadlocks and aborts one of the transactions with `ERROR: deadlock detected`. If the application does not catch this exception and retry, the user gets a 500 error for something that would succeed on a second attempt. Add deadlock retry logic alongside serialization failure retry logic.

**4. Using table-level locks when row-level locks suffice**
`LOCK TABLE accounts IN EXCLUSIVE MODE` blocks every reader and writer for the entire table for the duration of the transaction. `SELECT ... FOR UPDATE` locks only the specific rows you are modifying. Always use the most granular lock type that satisfies your correctness requirements.

**5. SELECT FOR UPDATE on a large result set**
`SELECT * FROM transactions WHERE status = 'pending' FOR UPDATE` locks every pending row. If there are 100,000 pending rows, this creates 100,000 row locks, blocks all other writers on pending transactions, and likely causes cascading deadlocks. Use `LIMIT` with `FOR UPDATE` and process in small batches.

**6. Assuming SQLite provides the same row-level locking as PostgreSQL**
SQLite has no row-level locking. In WAL mode, one writer holds a database-level write lock. Two concurrent Python threads both writing to SQLite will serialise automatically (one waits), which is correct but may cause unexpected latency. For high-concurrency write workloads, PostgreSQL's row-level locking is essential.


---
*sql_methods_library - Samantha McGarrigle*